# CatBoost Experiment

This experiment tests CatBoost on the EV purchase prediction dataset.

CatBoost can handle categorical features directly, giving us a different modeling approach from our XGBoost experiments.

Current best validation ROC-AUC: **0.941731**

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

train = pd.read_csv(project_root / 'data' / 'train.csv')

print('Train shape:', train.shape)

Train shape: (668665, 15)


In [2]:
X = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

categorical_features = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

categorical_indices = [
    X.columns.get_loc(column) for column in categorical_features
]

print('Training rows:', X_train.shape[0])
print('Validation rows:', X_valid.shape[0])
print('Categorical features:', categorical_features)

C:\Users\aakif\AppData\Local\Temp\ipykernel_12516\2026602061.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


Training rows: 534932
Validation rows: 133733
Categorical features: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [3]:
model = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=False,
    thread_count=-1
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_indices,
    eval_set=(X_valid, y_valid),
    verbose=False
)

valid_predictions = model.predict_proba(X_valid)[:, 1]
roc_auc = roc_auc_score(y_valid, valid_predictions)

print(f'CatBoost ROC-AUC: {roc_auc:.6f}')
print(f'Previous best ROC-AUC: {0.941731:.6f}')
print(f'Difference: {roc_auc - 0.941731:+.6f}')

CatBoost ROC-AUC: 0.941433
Previous best ROC-AUC: 0.941731
Difference: -0.000298


## Result

This is a validation experiment only. We will create a Kaggle submission only if CatBoost gives us a meaningful improvement over the current best XGBoost result.